# Chapter 11: Fine-tuning BERT - Easy Tasks

This notebook covers fundamental fine-tuning concepts: supervised classification, exploring model architecture, and making predictions.

## Setup

Run all cells in this section to set up the environment and load the data.

Before running these cells, review the concepts from the main Chapter 11 notebook.

### [Optional] - Installing Packages on Google Colab

If you are viewing this notebook on Google Colab, uncomment and run the following code to install dependencies.

**Note**: Use a GPU for this notebook. In Google Colab, go to Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4.

In [4]:
%%capture
!pip install "datasets>=2.18.0,<3" transformers>=4.38.2 accelerate>=0.27.2 evaluate

### Import Libraries

In [5]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import DataCollatorWithPadding
from transformers import TrainingArguments, Trainer
import numpy as np
import evaluate

### Load Data

In [6]:
# Load Rotten Tomatoes dataset
tomatoes = load_dataset("rotten_tomatoes")

Generating test split: 100%|██████████| 1066/1066 [00:00<00:00, 495690.47 examples/s]


In [7]:
# Prepare data and splits
train_data, test_data = tomatoes["train"], tomatoes["test"]

### Examine the Data

In [8]:
# View dataset structure
print("Dataset structure:")
print(tomatoes)

Dataset structure:
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})


In [9]:
# View first training example
print("First example:")
print(train_data[0])

First example:
{'text': 'the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .', 'label': 1}


In [10]:
# Check label distribution
print(f"\nTraining samples: {len(train_data)}")
print(f"Test samples: {len(test_data)}")


Training samples: 8530
Test samples: 1066


## Challenges

Complete the following tasks by implementing the starter code.

### Level: Easy

**About This Task:**

The HuggingFace Trainer provides a simple interface for fine-tuning pre-trained models. We'll train BERT to classify movie reviews as positive or negative.

#### Easy Task 1: Train a Classifier with HuggingFace Trainer

### Instructions

1. Load BERT model and tokenizer
2. Tokenize the training and test data
3. Define evaluation metrics (F1 score)
4. Create training arguments and trainer
5. Train the model and evaluate results

Load the pre-trained model and tokenizer.

In [11]:
# Load Model and Tokenizer
model_id = "bert-base-cased"
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)
tokenizer = AutoTokenizer.from_pretrained(model_id)

/media/danielcastillo/DanielSSD1/ads525/ads525/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Create a data collator for padding.

In [12]:
# Pad to the longest sequence in the batch
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Define a function to tokenize the text.

In [13]:
def preprocess_function(examples):
    """Tokenize input data"""
    return tokenizer(examples["text"], truncation=True)

Tokenize the training and test data.

In [14]:
# Tokenize train/test data
tokenized_train = train_data.map(preprocess_function, batched=True)
tokenized_test = test_data.map(preprocess_function, batched=True)

Map: 100%|██████████| 1066/1066 [00:00<00:00, 22993.83 examples/s]


View a tokenized example.

In [15]:
# Examine tokenized data
print("Tokenized example:")
print(tokenized_train[0])

Tokenized example:
{'text': 'the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .', 'label': 1, 'input_ids': [101, 1103, 2067, 1110, 17348, 1106, 1129, 1103, 6880, 1432, 112, 188, 1207, 107, 14255, 1389, 107, 1105, 1115, 1119, 112, 188, 1280, 1106, 1294, 170, 24194, 1256, 3407, 1190, 170, 11791, 5253, 188, 1732, 7200, 10947, 12606, 2895, 117, 179, 7766, 118, 172, 15554, 1181, 3498, 6961, 3263, 1137, 188, 1566, 7912, 14516, 6997, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


Define the evaluation metric.

In [16]:
def compute_metrics(eval_pred):
    """Calculate F1 score"""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    load_f1 = evaluate.load("f1")
    f1 = load_f1.compute(predictions=predictions, references=labels)["f1"]
    return {"f1": f1}

Define training arguments.

In [17]:
# Training arguments for parameter tuning
training_args = TrainingArguments(
    "model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    save_strategy="epoch",
    report_to="none"
)

Create the trainer.

In [18]:
# Trainer which executes the training process
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

Train the model.

In [19]:
# Start training
trainer.train()

 94%|█████████▍| 502/534 [00:57<00:03,  9.00it/s]

{'loss': 0.4064, 'grad_norm': 10.521149635314941, 'learning_rate': 1.2734082397003748e-06, 'epoch': 0.94}


100%|██████████| 534/534 [01:03<00:00,  8.36it/s]

{'train_runtime': 63.8842, 'train_samples_per_second': 133.523, 'train_steps_per_second': 8.359, 'train_loss': 0.4020274658774615, 'epoch': 1.0}


TrainOutput(global_step=534, training_loss=0.4020274658774615, metrics={'train_runtime': 63.8842, 'train_samples_per_second': 133.523, 'train_steps_per_second': 8.359, 'train_loss': 0.4020274658774615, 'epoch': 1.0})

Evaluate the trained model.

In [20]:
# Evaluate on test set
results = trainer.evaluate()
print(f"\nF1 Score: {results['eval_f1']:.4f}")

100%|██████████| 67/67 [00:03<00:00, 18.74it/s]


F1 Score: 0.8523


### Task 1a: Analyze Training Metrics

Look at the training output above and answer the questions.

### Questions

1. What is the F1 score of your trained model?

2. How long did training take? How many steps were executed?

3. What would happen if you increased num_train_epochs to 3? Would the F1 score improve?

**About This Task:**

Understanding a model's architecture helps you know which layers to freeze and how to customize fine-tuning.

#### Easy Task 2: Explore Model Architecture and Layers

### Instructions

1. Examine all parameter names in the model
2. Count how many encoder layers BERT has
3. Identify the classifier head parameters
4. Understand the naming convention for layers
5. Calculate total number of parameters

Reload a fresh model.

In [21]:
# Load a fresh model for exploration
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)
tokenizer = AutoTokenizer.from_pretrained(model_id)

/media/danielcastillo/DanielSSD1/ads525/ads525/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Print all layer names.

In [22]:
# Print all parameter names
print("All model parameters:")
for name, param in model.named_parameters():
    print(name)

All model parameters:
bert.embeddings.word_embeddings.weight
bert.embeddings.position_embeddings.weight
bert.embeddings.token_type_embeddings.weight
bert.embeddings.LayerNorm.weight
bert.embeddings.LayerNorm.bias
bert.encoder.layer.0.attention.self.query.weight
bert.encoder.layer.0.attention.self.query.bias
bert.encoder.layer.0.attention.self.key.weight
bert.encoder.layer.0.attention.self.key.bias
bert.encoder.layer.0.attention.self.value.weight
bert.encoder.layer.0.attention.self.value.bias
bert.encoder.layer.0.attention.output.dense.weight
bert.encoder.layer.0.attention.output.dense.bias
bert.encoder.layer.0.attention.output.LayerNorm.weight
bert.encoder.layer.0.attention.output.LayerNorm.bias
bert.encoder.layer.0.intermediate.dense.weight
bert.encoder.layer.0.intermediate.dense.bias
bert.encoder.layer.0.output.dense.weight
bert.encoder.layer.0.output.dense.bias
bert.encoder.layer.0.output.LayerNorm.weight
bert.encoder.layer.0.output.LayerNorm.bias
bert.encoder.layer.1.attention.self

Count encoder layers.

In [23]:
# Count encoder layers
encoder_layers = set()
for name, param in model.named_parameters():
    if "encoder.layer" in name:
        layer_num = name.split("layer.")[1].split(".")[0]
        encoder_layers.add(int(layer_num))

print(f"\nNumber of encoder layers: {len(encoder_layers)}")
print(f"Layer indices: {sorted(encoder_layers)}")


Number of encoder layers: 12
Layer indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]


Identify classifier parameters.

In [24]:
# Find classifier parameters
print("\nClassifier parameters:")
for name, param in model.named_parameters():
    if "classifier" in name:
        print(f"{name}: shape {param.shape}")


Classifier parameters:
classifier.weight: shape torch.Size([2, 768])
classifier.bias: shape torch.Size([2])


Calculate total parameters.

In [25]:
# Count total parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")


Total parameters: 108,311,810
Trainable parameters: 108,311,810


### Task 2a: Examine Layer Components

Each encoder layer has multiple components. Count them.

In [26]:
# Examine a single layer's components
print("Layer 0 components:")
for name, param in model.named_parameters():
    if "encoder.layer.0" in name:
        print(name)

Layer 0 components:
bert.encoder.layer.0.attention.self.query.weight
bert.encoder.layer.0.attention.self.query.bias
bert.encoder.layer.0.attention.self.key.weight
bert.encoder.layer.0.attention.self.key.bias
bert.encoder.layer.0.attention.self.value.weight
bert.encoder.layer.0.attention.self.value.bias
bert.encoder.layer.0.attention.output.dense.weight
bert.encoder.layer.0.attention.output.dense.bias
bert.encoder.layer.0.attention.output.LayerNorm.weight
bert.encoder.layer.0.attention.output.LayerNorm.bias
bert.encoder.layer.0.intermediate.dense.weight
bert.encoder.layer.0.intermediate.dense.bias
bert.encoder.layer.0.output.dense.weight
bert.encoder.layer.0.output.dense.bias
bert.encoder.layer.0.output.LayerNorm.weight
bert.encoder.layer.0.output.LayerNorm.bias


### Questions

1. How many encoder layers does BERT-base have?

2. What are the main components in each encoder layer? (attention, intermediate, output)

3. Why are the classifier weights randomly initialized while other weights are pre-trained?

**About This Task:**

After training, you can use the model to make predictions on new text. This is the ultimate goal of fine-tuning.

#### Easy Task 3: Make Predictions with Trained Model

### Instructions

1. Use the trained model to classify new reviews
2. Examine the prediction logits and probabilities
3. Test on positive and negative examples
4. Create your own test cases
5. Analyze model confidence scores

Define a prediction function.

In [27]:
import torch

def predict_sentiment(text):
    """Predict sentiment for a given text"""
    # Tokenize input
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    
    # Get predictions
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Get probabilities
    logits = outputs.logits
    probs = torch.nn.functional.softmax(logits, dim=-1)
    
    # Get prediction
    prediction = torch.argmax(logits, dim=-1).item()
    confidence = probs[0][prediction].item()
    
    label = "Positive" if prediction == 1 else "Negative"
    
    return label, confidence, probs[0].tolist()

Test on a positive review.

In [28]:
# Test case 1: Positive review
text1 = "This movie was absolutely fantastic! I loved every minute of it."

label, confidence, probs = predict_sentiment(text1)
print(f"Text: {text1}")
print(f"Prediction: {label}")
print(f"Confidence: {confidence:.4f}")
print(f"Probabilities [Neg, Pos]: {probs}")

Text: This movie was absolutely fantastic! I loved every minute of it.
Prediction: Negative
Confidence: 0.6833
Probabilities [Neg, Pos]: [0.6833423376083374, 0.3166576325893402]


Test on a negative review.

In [29]:
# Test case 2: Negative review
text2 = "Terrible film. Waste of time and money."

label, confidence, probs = predict_sentiment(text2)
print(f"\nText: {text2}")
print(f"Prediction: {label}")
print(f"Confidence: {confidence:.4f}")
print(f"Probabilities [Neg, Pos]: {probs}")


Text: Terrible film. Waste of time and money.
Prediction: Negative
Confidence: 0.6587
Probabilities [Neg, Pos]: [0.6586698889732361, 0.3413301408290863]


Test on a neutral/ambiguous review.

In [30]:
# Test case 3: Neutral review
text3 = "The movie was okay, nothing special."

label, confidence, probs = predict_sentiment(text3)
print(f"\nText: {text3}")
print(f"Prediction: {label}")
print(f"Confidence: {confidence:.4f}")
print(f"Probabilities [Neg, Pos]: {probs}")


Text: The movie was okay, nothing special.
Prediction: Negative
Confidence: 0.6804
Probabilities [Neg, Pos]: [0.6804472208023071, 0.31955277919769287]


### Task 3a: Test Your Own Examples

Create your own movie review texts and test them.

In [31]:
# Fill in: Your positive review
my_positive_review = "TODO: Write a positive movie review"

if "TODO" not in my_positive_review:
    label, confidence, probs = predict_sentiment(my_positive_review)
    print(f"Text: {my_positive_review}")
    print(f"Prediction: {label}")
    print(f"Confidence: {confidence:.4f}")

In [32]:
# Fill in: Your negative review
my_negative_review = "TODO: Write a negative movie review"

if "TODO" not in my_negative_review:
    label, confidence, probs = predict_sentiment(my_negative_review)
    print(f"\nText: {my_negative_review}")
    print(f"Prediction: {label}")
    print(f"Confidence: {confidence:.4f}")

Test on actual test set examples.

In [33]:
# Test on real examples from test set
for i in [0, 10, 20]:
    text = test_data[i]["text"]
    true_label = "Positive" if test_data[i]["label"] == 1 else "Negative"
    
    pred_label, confidence, probs = predict_sentiment(text)
    
    print(f"\nExample {i}:")
    print(f"Text: {text[:80]}...")
    print(f"True label: {true_label}")
    print(f"Predicted: {pred_label}")
    print(f"Confidence: {confidence:.4f}")
    print(f"Correct: {true_label == pred_label}")


Example 0:
Text: lovingly photographed in the manner of a golden book sprung to life , stuart lit...
True label: Positive
Predicted: Negative
Confidence: 0.6773
Correct: False

Example 10:
Text: exposing the ways we fool ourselves is one hour photo's real strength ....
True label: Positive
Predicted: Negative
Confidence: 0.6760
Correct: False

Example 20:
Text: this kind of hands-on storytelling is ultimately what makes shanghai ghetto move...
True label: Positive
Predicted: Negative
Confidence: 0.6831
Correct: False


### Questions

1. How does the model handle neutral/ambiguous reviews? Does it have high or low confidence?

2. What is the relationship between the two probability scores? (They should sum to 1)

3. When might low confidence predictions be useful to identify in a production system?